# OpenPlaque — Source-Led Long Proximal-Trunk Extension v1

Extends the dense-QC positive local multidirection continuation without using aortic geometry during discovery. Aortic distance is evaluated only post-hoc. Frozen master remains unchanged.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, datetime
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR=DRIVE_ROOT+'/Left_Proximal_Trunk_Source_Led_Long_Extension_v1'
REUSE_EXISTING_OUTPUTS=False
BRANCH='left-proximal-trunk-source-led-long-extension-from-main'
PINNED_SCIENCE_COMMIT='6df5436a70d5db2e6e1228ee102ad04ed755b583'
BASELINE='0593b453959f5a353d644267fbeef24b514ef4d7'
EXPECTED_ALGORITHM='left-proximal-trunk-source-led-long-extension-v1.0'
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
Path(OUTPUT_DIR,'notebook_started.json').write_text(json.dumps({'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'branch':BRANCH,'science_pin':PINNED_SCIENCE_COMMIT},indent=2))
print('Output:',OUTPUT_DIR)
print('Branch:',BRANCH)
print('Science pin:',PINNED_SCIENCE_COMMIT)


In [ ]:
import os, shutil, subprocess
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
subprocess.run(['git','clone','--depth','20','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',repo],check=True)
subprocess.run(['git','-C',repo,'checkout','--detach',PINNED_SCIENCE_COMMIT],check=True)
head=subprocess.check_output(['git','-C',repo,'rev-parse','HEAD'],text=True).strip()
mb=subprocess.check_output(['git','-C',repo,'merge-base','HEAD',BASELINE],text=True).strip()
print('HEAD:',head)
print('Merge base:',mb)
assert head==PINNED_SCIENCE_COMMIT,(head,PINNED_SCIENCE_COMMIT)
assert mb==BASELINE,(mb,BASELINE)


In [ ]:
%pip uninstall -y openplaque >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy pydicom psutil


In [ ]:
import sys, importlib, pathlib, pytest
for name in list(sys.modules):
    if name=='openplaque' or name.startswith('openplaque.'): del sys.modules[name]
importlib.invalidate_caches()
import openplaque
from openplaque import left_proximal_trunk_source_led_long_extension_v1 as exp
print('openplaque:',openplaque.__file__)
print('experiment:',exp.__file__)
print('algorithm:',exp.ALGORITHM)
assert exp.BASELINE==BASELINE
assert exp.ALGORITHM==EXPECTED_ALGORITHM
compile(pathlib.Path(exp.__file__).read_text(),exp.__file__,'exec')
print('synthetic:',exp.synthetic_long_extension_self_test())
rc=pytest.main(['-q','/content/OpenPlaque/tests/test_left_proximal_trunk_source_led_long_extension_v1.py'])
if rc!=0: raise RuntimeError(f'pytest failed: {rc}')


In [ ]:
from pathlib import Path
import json, datetime
root=Path(DRIVE_ROOT)
required=[
 root/'Left_Proximal_Trunk_Local_Multidirection_v1/summary.json',
 root/'Left_Proximal_Trunk_Local_Multidirection_v1/best_local_multidirection_path.csv',
 root/'Left_Proximal_Trunk_Continuation_QC_v1/accepted_proximal_trunk_continuation_candidate.csv',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/aorta.nii.gz']
missing=[str(p) for p in required if not p.exists()]
print('required:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))
prior=json.loads(required[0].read_text())
print('Prior status:',prior.get('status'))
print('Prior accepted hypotheses:',prior.get('n_accepted_hypotheses'))
assert prior.get('status')=='PROXIMAL_TRUNK_LOCAL_MULTIDIRECTION_CONTINUATION_QC_POSITIVE'
Path(OUTPUT_DIR,'preflight_complete.json').write_text(json.dumps({'completed_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'prior_status':prior.get('status')},indent=2))


In [ ]:
import time, gc
from openplaque.left_proximal_trunk_source_led_long_extension_v1 import run
gc.collect();t0=time.time()
result=run(DRIVE_ROOT,OUTPUT_DIR)
s=result['summary'];b=s.get('best_hypothesis',{});p=s.get('posthoc',{})
print('ELAPSED MIN:',round((time.time()-t0)/60,2))
print('STATUS:',s.get('status'))
print('RCA CONTROL PASS:',s.get('RCA_plane_qc_control',{}).get('accepted'))
print('HYPOTHESES:',s.get('n_hypotheses'),'ACCEPTED:',s.get('n_accepted_hypotheses'))
print('BEST ACCEPTED EXTENSION MM:',b.get('accepted_arc_mm'))
print('BEST PLANE PASS FRACTION:',b.get('accepted_plane_pass_fraction'))
print('AORTA START/MIN/END MM:',p.get('start_aorta_distance_mm'),p.get('minimum_aorta_distance_mm'),p.get('end_aorta_distance_mm'))
print('EVENTUALLY TURNS TOWARD AORTA:',p.get('eventually_turns_toward_aorta'))
print('REACHES AORTA:',p.get('reaches_aorta'))
print('REPORT:',result.get('report'))
print('ZIP:',result.get('zip'))
